# Environment / Dependencies Setup
Installs NetworkX (Part 1 in-memory graph), the Neo4j driver + visualizer (Part 2), PDF/download helpers, and dotenv.

In [ ]:
!pip install networkx neo4j requests PyPDF2 yfiles-jupyter-graphs-for-neo4j python-dotenv

## Import Libraries

In [ ]:
import os
import json
import requests
import PyPDF2
import networkx as nx
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Load the secrets (OPENROUTER_API_KEY, NEO4J_*) from the .env file next to this notebook
load_dotenv(".env")


## Configure the LLM

In [ ]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()
else:
    print("Success: API Key loaded!")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
TEXT_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"


def ask_llm(prompt):
    """Send one prompt to the LLM and return its text reply (used by every step)."""
    payload = {
        "model": TEXT_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0,
        # This model "thinks" before answering; turning that off stops it from
        # using up its whole token budget on hidden reasoning and returning nothing.
        "reasoning": {"enabled": False},
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload)
    resp.raise_for_status()
    # the reply text sits on the usual OpenAI path: choices[0] -> message -> content
    return resp.json()["choices"][0]["message"]["content"].strip()

The key is read from the `.env` file loaded above, or typed in if missing. `ask_llm` is the single reused helper for extraction and answering.

In [ ]:
PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"

os.makedirs("data", exist_ok=True)
PDF_PATH = os.path.join("data", "document.pdf")

try:
    print("Downloading document...")
    response = requests.get(PDF_URL)
    response.raise_for_status()

    with open(PDF_PATH, "wb") as f:
        f.write(response.content)

    print(f"Success! PDF stored at: {PDF_PATH}")
except Exception as e:
    print(f"Error downloading PDF: {e}")

By the end of this step the PDF exists at `data/document.pdf`. Nothing has been read from it yet.

In [ ]:
with open(PDF_PATH, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    sample_text = reader.pages[0].extract_text()

sample_text = sample_text[:1500].strip()
print(f"Success! Extracted {len(sample_text)} characters.")

`sample_text` now holds a short block of plain text (first 1500 chars of page 1), ready for the LLM.

In [ ]:
EXTRACT_PROMPT = """
You are an expert knowledge graph builder. Read the text below and extract all key entities and their relationships.

Text:
{text}

CRITICAL INSTRUCTIONS:
Output ONLY a valid JSON list of objects with keys "source", "relation", and "target".
Example format:
[
  {{"source": "Entity_A", "relation": "RELATES_TO", "target": "Entity_B"}},
  {{"source": "Entity_C", "relation": "CAUSES", "target": "Entity_D"}}
]
Do not add any Markdown code blocks, explanations, or introductory text. Return ONLY pure JSON.
"""


def extract_graph_elements(text):
    """Use the LLM to turn raw text into a list of source/relation/target triples."""
    try:
        raw_json = ask_llm(EXTRACT_PROMPT.format(text=text))

        # Some models wrap the JSON in a ``` fence; strip it if present.
        if raw_json.startswith("```"):
            raw_json = raw_json.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        return json.loads(raw_json)
    except Exception as e:
        print(f"Extraction Error: {e}")
        return []

Run it **once** and inspect the result:

In [ ]:
extracted_relationships = extract_graph_elements(sample_text)
print(f"Extracted {len(extracted_relationships)} relationships:\n")
print(json.dumps(extracted_relationships, indent=2))

`extracted_relationships` is a plain Python list of triples used by both parts, so the LLM only extracts once. You should see a count and the JSON list.

In [ ]:
G = nx.DiGraph()

for rel in extracted_relationships:
    src = rel["source"].strip()
    tgt = rel["target"].strip()
    relation_type = rel["relation"].strip()

    G.add_edge(src, tgt, relation=relation_type)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

`add_edge` creates any missing node automatically. You should see a node count and an edge count.

In [ ]:
def find_node_in_question(question, node_names):
    """Return the known entity whose name appears in the question (longest match wins)."""
    matches = [n for n in node_names if n.lower() in question.lower()]
    return max(matches, key=len) if matches else None

It takes a plain list of names, so Part 2 can reuse it with names read from Neo4j.

In [ ]:
def traverse_subgraph(graph, start_node, radius=2):
    """Collect every relationship within `radius` hops of a node, as readable facts."""
    # Breadth-first walk: every node reachable from start_node in <= radius hops
    reachable = nx.single_source_shortest_path_length(graph, start_node, cutoff=radius).keys()
    subgraph = graph.subgraph(reachable)

    facts = []
    for u, v, data in subgraph.edges(data=True):
        facts.append(f"{u} --[{data.get('relation', 'CONNECTED_TO')}]--> {v}")
    return facts

`single_source_shortest_path_length(cutoff=radius)` walks outward from the start node and stops after `radius` hops.

```mermaid
flowchart LR
    T[Transformer]
    A[Attention Mechanism<br/>answer found here]
    E[Encoder]
    S[Self-Attention<br/>2nd hop, extra context]
    N[Neural Machine Translation<br/>3rd hop, never reached]

    T -->|hop 1| A
    T -->|hop 1| E
    A -->|hop 2| S
    S -.->|too far| N

    class T,A,S hop
    classDef hop fill:#ffe08a,stroke:#d68f00,stroke-width:2px,color:#1a1a1a;

```

In [ ]:
QA_TEMPLATE = """
You are an expert AI research assistant using a Knowledge Graph.
Answer the question using ONLY the connected relationship paths provided below.

Graph Relationships:
{facts_block}

Question: {question}

CRITICAL INSTRUCTIONS:
Output your response in EXACTLY two sections as shown below.

--- FINAL ANSWER ---
[Provide a direct, simple, 1-sentence answer.]

--- AI TRACING & EXPLAINABILITY ---
[Explain step-by-step how the answer was derived from the graph. Use an objective, third-person perspective. Do NOT use first-person pronouns like "I" or "my".]
"""


def answer_from_facts(question, facts):
    """Ask the LLM to answer using only the given facts, with a reasoning trace."""
    facts_block = "\n".join(f"- {f}" for f in facts)
    try:
        return ask_llm(QA_TEMPLATE.format(facts_block=facts_block, question=question))
    except Exception as e:
        return f"Error executing Graph RAG: {e}"

The in-memory pipeline is then three lines of logic: find the node, walk the graph, answer.

In [ ]:
def execute_graph_rag(question):
    """Part 1 pipeline: find the node, walk the NetworkX graph, answer."""
    target_entity = find_node_in_question(question, G.nodes)
    if not target_entity:
        return f"Could not find any known concepts in your question. Known concepts: {list(G.nodes)[:5]}..."

    print(f"[System Log] Auto-detected focus concept: '{target_entity}'")
    facts = traverse_subgraph(G, target_entity, radius=2)
    if not facts:
        return f"Found the concept '{target_entity}', but no relationships are connected to it."

    return answer_from_facts(question, facts)

If no entity matches, the function stops and lists some known ones instead of guessing.

---

### Step 7 — Test Part 1

In [ ]:
QUESTION = "What mechanism does the Transformer architecture rely on?"
print(execute_graph_rag(QUESTION))

### Step 7 — Test Part 1
You should see the detected concept, a `--- FINAL ANSWER ---`, and a `--- AI TRACING ---` section. Part 1's graph dies with the session; Part 2 puts the same triples into Neo4j.

In [ ]:
# Values come from the .env file loaded in the imports step
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Success: Connected to Neo4j Database!")


> **Note:** `NEO4J_URI`, `NEO4J_USERNAME` and `NEO4J_PASSWORD` are read from the `.env` file. `verify_connectivity()` fails right here if they are wrong.

In [ ]:
def insert_into_neo4j(tx, relationships):
    for rel in relationships:
        source = rel["source"].strip()
        target = rel["target"].strip()

        # Neo4j relationship types are UPPERCASE_WITH_UNDERSCORES (e.g. HAS_AUTHOR)
        relation_type = rel["relation"].strip().replace(" ", "_").replace("-", "_").upper()

        query = f"""
        MERGE (s:Concept {{name: $source}})
        MERGE (t:Concept {{name: $target}})
        MERGE (s)-[:{relation_type}]->(t)
        """
        tx.run(query, source=source, target=target)


with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")  # start from an empty database
    print("Database cleared for fresh ingestion.")

    session.execute_write(insert_into_neo4j, extracted_relationships)
    print("Success: Knowledge Graph loaded into Neo4j!")

Each triple becomes two `Concept` nodes and one relationship, written with `MERGE` so repeated entities are never duplicated.

In [ ]:
with driver.session() as session:
    # .single() returns the one row; ["c"] reads the column aliased AS c
    n_nodes = session.run("MATCH (n:Concept) RETURN count(n) AS c").single()["c"]
    n_edges = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
print(f"Neo4j now holds {n_nodes} nodes and {n_edges} relationships.")

`$source`/`$target` are Cypher query parameters (relationship *types* cannot be). You should see counts matching the NetworkX graph from Step 4.

In [ ]:
def fetch_node_names():
    """Read every Concept name stored in Neo4j."""
    with driver.session() as session:
        return [record["name"] for record in session.run("MATCH (c:Concept) RETURN c.name AS name")]


def fetch_graph_context(entity_name, hops=2):
    """Pull every relationship within `hops` steps of a node, as readable facts."""
    # Cypher cannot take the path length as a parameter, so it is placed in the text
    # (int() makes sure only a number can ever get in).
    query = f"""
    MATCH p = (n:Concept {{name: $name}})-[*1..{int(hops)}]-(m:Concept)
    UNWIND relationships(p) AS r
    RETURN DISTINCT startNode(r).name AS source, type(r) AS relation, endNode(r).name AS target
    """
    with driver.session() as session:
        result = session.run(query, name=entity_name)
        return [f"{rec['source']} --[{rec['relation']}]--> {rec['target']}" for rec in result]

The same two jobs as Step 5, done by the database. `[*1..2]` is the radius-2 walk; `UNWIND` splits each path into its individual relationships.

In [ ]:
focus = find_node_in_question(QUESTION, fetch_node_names())
print(f"Focus node: {focus}")
for fact in fetch_graph_context(focus, hops=1):
    print(" ", fact)

You should see the focus node (`Transformer`) and its direct 1-hop facts as `A --[TYPE]--> B` lines.

In [ ]:
def execute_neo4j_rag(question):
    """Part 2 pipeline: find the node in Neo4j, fetch its facts with Cypher, answer."""
    target_entity = find_node_in_question(question, fetch_node_names())
    if not target_entity:
        return "Could not find any known concepts from the database in your question."

    print(f"[System Log] Auto-detected focus concept: '{target_entity}'")
    facts = fetch_graph_context(target_entity, hops=2)
    if not facts:
        return f"Found '{target_entity}' in Neo4j, but no relationships were connected to it."

    return answer_from_facts(question, facts)


print(execute_neo4j_rag(QUESTION))

### Step 11 — Answer the Same Question from Neo4j
Only retrieval changes — the Step 6 answering function is reused unchanged.

In [ ]:
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget

widget = Neo4jGraphWidget(driver)
widget.show_cypher("MATCH (n)-[r]->(m) RETURN n, r, m")

`show_cypher` draws the query result as a zoomable, clickable graph. You should see the nodes and labelled arrows; close the driver afterward.

In [ ]:
driver.close()

### What We Learnt
The same triples in two graph stores — walked in memory with NetworkX, then read with Cypher from Neo4j — each time with a written trail of the relationships followed.